In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
data = pd.read_csv("../data/telco_customer_churn.csv")

In [3]:
data.info()
data["Churn"] = data["Churn"].map({"Yes":1, "No":0})

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)
for train_index, test_index in split.split(data, data["Churn"]):
    train_set = data.loc[train_index]
    test_set = data.loc[test_index]

train_label = train_set["Churn"].copy()
test_label =  test_set["Churn"].copy()
train_nl = train_set.drop("Churn", axis = 1)
test_nl = test_set.drop("Churn", axis = 1)


In [5]:
train_set.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 3738 to 5639
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   str    
 1   gender            5634 non-null   str    
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   str    
 4   Dependents        5634 non-null   str    
 5   tenure            5634 non-null   int64  
 6   PhoneService      5634 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          5634 non-null   str    
 16  PaperlessBilling  5634 non-null   str    
 17  PaymentM

In [6]:
test_set.info()

<class 'pandas.DataFrame'>
Index: 1409 entries, 437 to 5613
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        1409 non-null   str    
 1   gender            1409 non-null   str    
 2   SeniorCitizen     1409 non-null   int64  
 3   Partner           1409 non-null   str    
 4   Dependents        1409 non-null   str    
 5   tenure            1409 non-null   int64  
 6   PhoneService      1409 non-null   str    
 7   MultipleLines     1409 non-null   str    
 8   InternetService   1409 non-null   str    
 9   OnlineSecurity    1409 non-null   str    
 10  OnlineBackup      1409 non-null   str    
 11  DeviceProtection  1409 non-null   str    
 12  TechSupport       1409 non-null   str    
 13  StreamingTV       1409 non-null   str    
 14  StreamingMovies   1409 non-null   str    
 15  Contract          1409 non-null   str    
 16  PaperlessBilling  1409 non-null   str    
 17  PaymentMe

In [7]:
compare = pd.DataFrame({
    "FULL":data["Churn"].value_counts(normalize=True),
    "TRAIN":train_set["Churn"].value_counts(normalize=True),
    "TEST":test_set["Churn"].value_counts(normalize=True),
})
print(compare)

          FULL     TRAIN      TEST
Churn                             
0      0.73463  0.734647  0.734564
1      0.26537  0.265353  0.265436


In [8]:
print(train_set[train_set["InternetService"] == "No"]["DeviceProtection"].unique())
print(train_set[train_set["Dependents"] == "Yes"][["DeviceProtection", "OnlineSecurity", "OnlineBackup"]])

<StringArray>
['No internet service']
Length: 1, dtype: str
         DeviceProtection       OnlineSecurity         OnlineBackup
3151                   No                  Yes                   No
4860                   No                  Yes                  Yes
3810                   No                   No                   No
2666                  Yes                   No                  Yes
6950                   No                  Yes                   No
...                   ...                  ...                  ...
58    No internet service  No internet service  No internet service
608                   Yes                  Yes                  Yes
4332                   No                  Yes                   No
4635  No internet service  No internet service  No internet service
4546                  Yes                   No                  Yes

[1679 rows x 3 columns]


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [10]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin

In [12]:
class StrToNum():
    def __init__(self):
        pass
    # fit method is needed so that the model can learn something from the data
    def fit(self, X, y = None):
        return self
    def transform(self, X, y = None):
        #print(y)
        Y = X.copy()
        Y = Y.drop("customerID", axis = "columns")
        Y["TotalCharges"] = pd.to_numeric(Y["TotalCharges"], errors="coerce")
        Y = Y.replace("No phone service", "No" )
        Y = Y.replace("No internet service", "No")
        return Y

In [13]:
class C_OneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    # fit method is needed so that the model can learn something from the data
    def fit(self, X, y = None):
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        for column in self.columns:
            Y = pd.get_dummies(Y, columns = [column], dtype=int)
            
            if column + "_No internet service" in Y:
                Y = Y.drop(column + "_No internet service", axis = "columns")
        return Y

In [14]:
class OrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, values, multiple_values = False, dtype = int):
        self.columns = columns
        self.values = values
        self.multiple_values = multiple_values
        self.dtype = dtype
    def fit(self, X, y = None):
        self.fitted_ = True
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        for i in range(len(self.columns)):
            
            if self.multiple_values:
                Y[self.columns[i]] = Y[self.columns[i]].map(self.values[i]).astype(self.dtype)
            else:
                mapped = Y[self.columns[i]].map(self.values)
                unmapped = Y[self.columns[i]][mapped.isna()]
                #print(unmapped)
                Y[self.columns[i]] = Y[self.columns[i]].map(self.values).astype(self.dtype)
        return Y

In [15]:
from scipy.stats import chi2_contingency

def chi_square(x,y, N, alpha):
    result = {}
    table = pd.crosstab(x, y)
    chi2_stat, p_value, _, _ = chi2_contingency(table)
    cramers_v = np.sqrt(chi2_stat/(N*(min(table.shape) - 1)))

    return cramers_v
    

In [16]:
class DropFeature(BaseEstimator, TransformerMixin):
    def __init__(self,ignore, threshold):
        self.ignore = ignore
        self.threshold = threshold

    def fit(self, X, y = None):
        # y = train label
        Y = X.copy()
        
        self.result = []
        self.N = len(Y)
        
        self.all_columns_ = Y.select_dtypes(include = "int").columns
        
        self.to_drop_ = []
        for column in self.all_columns_:
            if column in self.ignore:
                continue
            cramers_v = chi_square(Y[column], y, self.N, 0.05)
            self.result.append({"column":column, "cramers_v":cramers_v})
            if cramers_v < self.threshold:
                self.to_drop_.append(column)
        self.fitted_ = True
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        for column in self.to_drop_:
            if column in Y:
                Y = Y.drop(column, axis=1)
       
        #df = pd.DataFrame(self.result)
        #df = df.sort_values(by="cramers_v")
        #print(df.T.to_string())
        
        return Y
        

In [17]:
class AddFeature(BaseEstimator, TransformerMixin):
    def __init__(self, add_Loyalty, add_ExtraService, add_ChargePerService, add_EntertainmentService, add_ProtectionService, add_Vunerable):
        self.add_Loyalty = add_Loyalty
        self.add_ExtraService = add_ExtraService
        self.add_ChargePerService = add_ChargePerService
        self.add_EntertainmentService = add_EntertainmentService
        self.add_ProtectionService = add_ProtectionService
        self.add_Vunerable = add_Vunerable

    def fit(self, X, y = None):
        self.fitted_ = True
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        Services = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                        "StreamingTV", "StreamingMovies", "PhoneService", 
                        "MultipleLines"]
        service_count = 0
        if self.add_Loyalty:
            Y["Loyalty"] = pd.cut(Y["tenure"], bins=[0, 12, 24, 48, 72], labels=[0, 1, 2, 3], include_lowest = True).astype(int)
        if self.add_ExtraService:
            Y["ExtraService"] = Y[Services].sum(axis = 1) + (Y["InternetService_Fiber optic"]| Y["InternetService_DSL"])

        if self.add_ChargePerService and "ExtraService" in Y.columns:
            Y["ChargePerService"] = Y["MonthlyCharges"]/Y["ExtraService"].replace(0,1)

        if self.add_EntertainmentService:
            Y["EntertainmentService"] = (Y["StreamingTV"] | Y["StreamingMovies"])
        if self.add_ProtectionService:
            Y["ProtectionService"] = Y["OnlineSecurity"] + Y["OnlineBackup"] + Y["DeviceProtection"]

        if self.add_Vunerable:
            Y["Vunerable"] = (Y["SeniorCitizen"] == 1) + (Y["Dependents"] == 0) + (Y["Partner"] == 0) + (Y["PaymentMethod_Electronic check"] == 1) + (Y["TechSupport"] == 0).astype(int)
       
        return Y

In [18]:
from scipy.stats import kstest, norm, zscore

In [19]:
def remove_outlier(df, column):
    data = df[column].dropna()
    mean = data.mean()
    std = data.std()

    print(mean)
    print(std)
    
    print(type(mean))
    print(type(std))
    
    stat, p = kstest(df[column], "norm", args=(mean, std))

    if p > 0.05:
        # apply z-score method
        lower = mean - 3*std
        upper = mean + 3*std
        z = zscore(df[column])
        df[column] = df[column].clip(lower = lower, upper = upper)
    else:
        # apply inter-quartile range method

        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        IQR = q3 - q1
        lower = q1 - 1.5*IQR
        upper = q3 + 1.5*IQR

    return (lower, upper)

In [20]:
class RemoveOutlier(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.range = []
        
    def fit(self, X, y = None):
        Y = X.copy()
        for column in self.columns:
                if column in Y.columns:
                    self.range.append(remove_outlier(Y, column))
        self.fitted_ = True
        return self

    def transform(self, X, y = None):
        Y = X.copy()

        for i in range(len(self.columns)):
            #print(self.columns, i)
            column = self.columns[i]
            #print(column, self.range)
            if column in Y.columns:
                Y[column] = Y[column].clip(lower = self.range[i][0], upper = self.range[i][1])

        return Y
    

In [21]:
from xgboost import XGBClassifier

In [22]:
columns = ["PaymentMethod", "InternetService"]
drop_columns = [
    "StreamingTV_No", "StreamingMovies_No", "OnlineSecurity_No",
    "OnlineSecurity_Yes", "OnlineBackup_No", "DeviceProtection_No"]

pipeline = Pipeline([
    ("to_num", StrToNum()),
    ("one_hot", C_OneHotEncoder(columns)),
    ("ordinal1", OrdinalEncoder(["Contract", "gender"],[{"Month-to-month":0, "One year":1, "Two year":2}, {"Male":1, "Female":0}], True)),
    ("ordinal2", OrdinalEncoder(["Partner",
                                  "Dependents","PaperlessBilling",
                                  "PhoneService", "MultipleLines",
                                  "OnlineSecurity", "OnlineBackup", 
                                  "DeviceProtection", "TechSupport",
                                 "StreamingTV", "StreamingMovies"],
                                {"Yes":1, "No":0}, False)),
    ("new_feature", AddFeature(True, True, True, True, True, True)),
    ("drop", DropFeature(["tenure"], 0.1)),
    #("remove_outlier", RemoveOutlier(["ChargePerService"])),
    ("xgb", XGBClassifier(random_state=42, eval_metric="auc")),
])

In [23]:
train_copy = train_nl.copy()

In [24]:
pipeline.fit(train_copy, train_label)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('to_num', ...), ('one_hot', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,columns,"['PaymentMethod', 'InternetService']"
,columns,"['Contract', 'gender']"
,values,"[{'Month-to-month': 0, 'One year': 1, 'Two year': 2}, {'Female': 0, 'Male': 1}]"
,multiple_values,True
,dtype,<class 'int'>


In [25]:
#new_train = pipeline.transform(train_copy)

In [26]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV

In [27]:
neg, pos = (train_label == 0).sum(), (train_label == 1).sum()
val = neg/pos
print(val)

2.768561872909699


In [28]:
skf = StratifiedKFold(n_splits = 5, random_state= 42, shuffle = True)


In [29]:
# ran_for = RandomForestClassifier(n_estimators = 100, max_depth = 5)

# model = XGBClassifier(n_estimators = 200, random_state = 42)
# model.fit(new_train, train_label)

# importances = pd.Series(model.feature_importances_, index= new_train.columns)
# importances.sort_values().plot(kind="barh", figsize=(8,10))

In [30]:
# stage1 = {
#     'xgb__n_estimators': [200, 300, 500, 1000],
#     'xgb__max_depth': [3, 6, 8],
#     'xgb__learning_rate':[0.01, 0.05, 0.1],
# }

# grid1 = GridSearchCV(pipeline, stage1, cv = skf, scoring = "roc_auc", return_train_score = True, n_jobs=-1)
# grid1.fit(train_copy, train_label)

# print(grid1)
# best = grid1.best_params_
# stage2={
#     'xgb__n_estimators':[best['xgb__n_estimators']],
#     'xgb__max_depth':[best['xgb__max_depth']],
#     'xgb__learning_rate':[best['xgb__learning_rate']],
#     'xgb__subsample':[0.3, 0.6, 0.8, 1],
#     'xgb__colsample_bytree':[0.3, 0.6, 0.8, 1],
# }

# grid2 = GridSearchCV(pipeline, stage2, cv = skf, scoring = "roc_auc", return_train_score = True, n_jobs = - 1)
# grid2.fit(train_copy, train_label)

# best = grid2.best_params_
# stage3 = {
#     'xgb__n_estimators':[best['xgb__n_estimators']],
#     'xgb__max_depth':[best['xgb__max_depth']],
#     'xgb__learning_rate':[best['xgb__learning_rate']],
#     'xgb__subsample':[best['xgb__subsample']],
#     'xgb__colsample_bytree':[best['xgb__colsample_bytree']],
#     'xgb__reg_alpha':[0,0.3,0.6, 1],
#     'xgb__reg_lambda':[1,3,5],
# }
# grid3 = GridSearchCV(pipeline, stage3, cv = skf, scoring = "roc_auc", return_train_score = True, n_jobs = -1)
# grid3.fit(train_copy, train_label)

# best = grid3.best_params_
# stage4 = {
#     'xgb__n_estimators':[best['xgb__n_estimators']],
#     'xgb__max_depth':[best['xgb__max_depth']],
#     'xgb__learning_rate':[best['xgb__learning_rate']],
#     'xgb__subsample':[best['xgb__subsample']],
#     'xgb__colsample_bytree':[best['xgb__colsample_bytree']],
#     'xgb__reg_alpha':[best['xgb__reg_alpha']],
#     'xgb__reg_lambda':[best['xgb__reg_lambda']],

#     'drop__threshold':[0.05, 0.08,  0.1, 0.12],
#     'new_feature__add_Loyalty':[True, False],
#     'new_feature__add_ExtraService':[True, False],
#     'new_feature__add_ChargePerService':[True, False],
#     'new_feature__add_EntertainmentService':[True, False],
#     'new_feature__add_ProtectionService':[True, False],
#     'new_feature__add_Vunerable':[True, False],
# }
# grid4 = RandomizedSearchCV(pipeline, stage4, n_iter=200, cv = skf, scoring = "roc_auc", return_train_score = True, n_jobs= -1)
# grid4.fit(train_copy, train_label)

In [31]:
parameter= [{
    'xgb__n_estimators': [100, 200, 300, 500, 1000],
    'xgb__max_depth':[3, 5, 8],
    'xgb__learning_rate':[0.01, 0.05, 0.1],
    'xgb__subsample':[0.3, 0.6, 0.8, 1],
    'xgb__colsample_bytree':[0.3, 0.6, 0.8, 1],
    'xgb__scale_pos_weight':[val, 2*val, 3*val],
    'xgb__reg_alpha':[0, 0.3, 0.6, 1],
    'xgb__reg_lambda':[1, 3, 5],

    #'drop__threshold':[0.05, 0.08,  0.1, 0.12],
    'new_feature__add_Loyalty':[True, False],
    'new_feature__add_ExtraService':[True, False],
    'new_feature__add_ChargePerService':[True, False],
    'new_feature__add_EntertainmentService':[True, False],
    'new_feature__add_ProtectionService':[True, False],
    'new_feature__add_Vunerable':[True, False],
}, ]
gridSearch = RandomizedSearchCV(pipeline, parameter, n_iter = 500, cv = skf, scoring = "roc_auc", return_train_score = True, n_jobs=-1, error_score='raise')
gridSearch.fit(train_copy, train_label)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'new_feature__add_ChargePerService': [True, False], 'new_feature__add_EntertainmentService': [True, False], 'new_feature__add_ExtraService': [True, False], 'new_feature__add_Loyalty': [True, False], ...}]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",500
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric ev

In [32]:
print(gridSearch.best_score_)

0.8491072683635024


In [33]:
print(gridSearch.best_params_)

{'xgb__subsample': 0.3, 'xgb__scale_pos_weight': np.float64(2.768561872909699), 'xgb__reg_lambda': 5, 'xgb__reg_alpha': 1, 'xgb__n_estimators': 1000, 'xgb__max_depth': 3, 'xgb__learning_rate': 0.01, 'xgb__colsample_bytree': 0.8, 'new_feature__add_Vunerable': False, 'new_feature__add_ProtectionService': False, 'new_feature__add_Loyalty': True, 'new_feature__add_ExtraService': False, 'new_feature__add_EntertainmentService': False, 'new_feature__add_ChargePerService': False}
